# 03 - Bake the ablation into the weights
**Paper:** Section 4. **Exercise used:** 5.
If nothing writes along `r_hat`, the residual stream never contains it. Editing ~2 matrices per layer gives a model that doesn't refuse, with no hooks.

In [ ]:
# --- Colab setup: run once per session ---
# 1) Runtime > Change runtime type > T4 GPU (or better)
# 2) Edit REPO to your GitHub URL (if the repo is private, use https://<TOKEN>@github.com/...)
REPO = "https://github.com/YOUR_GITHUB_USERNAME/refusal-direction"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
%pip install -q -e .
# If pip downgraded numpy and Colab asks you to restart: Runtime > Restart session, then rerun this cell.

In [ ]:
try:  # autoreload picks up edits to src/ without restarting
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:  # Colab's IPython autoreload imports `imp`, which Python 3.12+ removed
    pass
import torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config

# Save artifacts to Google Drive so they survive the Colab session ending.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/refusal-direction/artifacts"
except ImportError:
    ARTIFACTS = "artifacts"
cfg = Config(artifact_dir=ARTIFACTS)
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

In [ ]:
import copy
from refusal_direction import artifacts
from refusal_direction.interventions import get_ablation_hooks
from refusal_direction.orthogonalize import orthogonalize_weights
direction, meta = artifacts.load_direction(cfg)
direction = direction.to(model.cfg.device)

edited = copy.deepcopy(model)          # ~3 GB extra in fp16; fits on a T4
orthogonalize_weights(edited, direction)

## Equivalence check: edited model vs. hook-ablated model

In [ ]:
H = tok(data["harmful_test"][:16])
hooked = last_token_logits(model, *H, fwd_hooks=get_ablation_hooks(model, direction))
edit = last_token_logits(edited, *H)
print("max |logit diff|:", (hooked - edit).abs().max().item())
print("top-1 agreement:", (hooked.argmax(-1) == edit.argmax(-1)).float().mean().item())
# Not exactly 0 in fp16/bf16: the two compute the same function in a different order.

In [ ]:
harm = data["harmful_test"]
outs = generate(edited, *tok(harm), max_new_tokens=cfg.max_new_tokens, batch_size=cfg.batch_size)
rate = refusal_rate(outs)
artifacts.save_json(cfg, "orthogonalized_refusal_rate.json", {"harmful/orthogonalized": rate})
print("refusal rate (orthogonalized):", rate)

In [ ]:
for instr, out in list(zip(*(harm, outs)))[:3]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

## Questions to write up
1. How many parameters did you change, as a fraction of the model? What does it mean that a rank-1 edit removes a safety behavior?
2. The edited model's weights could be exported and shared. What does this imply for open-weight release policy? (Paper, Section 7 / ethics statement.)
3. Stretch: write a function that copies the edited TransformerLens weights back into a HF `AutoModelForCausalLM` and save it locally.